# 04 - GCC Regional Comparison

## Oman Economic Analysis Project

This notebook provides a comprehensive comparison of Oman with other GCC countries across multiple economic dimensions.

### Countries Analyzed
- Oman
- Saudi Arabia
- United Arab Emirates
- Qatar
- Kuwait
- Bahrain

In [ ]:
# Import libraries
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_processor import DataProcessor

plt.style.use('seaborn-v0_8-whitegrid')
%matplotlib inline

In [ ]:
# Load data
processor = DataProcessor(data_dir='../data')
df = processor.load_data('gcc_economic_data.csv')
df = processor.clean_data(df)

# Color scheme
colors = {
    'Oman': '#E41A1C',
    'Saudi Arabia': '#377EB8',
    'United Arab Emirates': '#4DAF4A',
    'Qatar': '#984EA3',
    'Kuwait': '#FF7F00',
    'Bahrain': '#A65628'
}

print(f"Loaded {len(df):,} records")

## 1. Economic Size Comparison

In [ ]:
# GDP comparison (latest year)
gdp_ind = 'NY.GDP.MKTP.CD'
latest_year = df[df['indicator_code'] == gdp_ind]['year'].max()

gdp_data = df[(df['indicator_code'] == gdp_ind) & (df['year'] == latest_year)].copy()
gdp_data['gdp_billions'] = gdp_data['value'] / 1e9
gdp_data = gdp_data.sort_values('gdp_billions', ascending=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart
ax1 = axes[0]
bar_colors = [colors.get(c, '#999999') for c in gdp_data['country_name']]
bars = ax1.barh(gdp_data['country_name'], gdp_data['gdp_billions'], color=bar_colors)
ax1.set_xlabel('GDP (Billion USD)', fontsize=12)
ax1.set_title(f'GCC GDP Comparison ({latest_year})', fontsize=14, fontweight='bold')
for bar, value in zip(bars, gdp_data['gdp_billions']):
    ax1.text(bar.get_width() + 5, bar.get_y() + bar.get_height()/2,
             f'${value:.0f}B', va='center', fontsize=10)

# Pie chart
ax2 = axes[1]
pie_colors = [colors.get(c, '#999999') for c in gdp_data['country_name']]
explode = [0.1 if c == 'Oman' else 0 for c in gdp_data['country_name']]
ax2.pie(gdp_data['gdp_billions'], labels=gdp_data['country_name'], 
        autopct='%1.1f%%', colors=pie_colors, explode=explode)
ax2.set_title('GCC GDP Share', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('../outputs/charts/gcc_gdp_overview.png', dpi=150, bbox_inches='tight')
plt.show()

# Oman's share
oman_share = gdp_data[gdp_data['country_name'] == 'Oman']['gdp_billions'].values[0] / gdp_data['gdp_billions'].sum() * 100
print(f"\nOman's share of GCC GDP: {oman_share:.1f}%")

## 2. GDP Per Capita Ranking

In [ ]:
# GDP per capita
gdp_pc_ind = 'NY.GDP.PCAP.CD'

gdp_pc_data = df[(df['indicator_code'] == gdp_pc_ind) & (df['year'] == latest_year)].copy()
gdp_pc_data = gdp_pc_data.sort_values('value', ascending=True)

fig, ax = plt.subplots(figsize=(10, 6))

bar_colors = [colors.get(c, '#999999') for c in gdp_pc_data['country_name']]
bars = ax.barh(gdp_pc_data['country_name'], gdp_pc_data['value'], color=bar_colors)

ax.set_xlabel('GDP per Capita (USD)', fontsize=12)
ax.set_title(f'GCC GDP per Capita ({latest_year})', fontsize=14, fontweight='bold')
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'${x/1000:.0f}K'))

for bar, value in zip(bars, gdp_pc_data['value']):
    if pd.notna(value):
        ax.text(bar.get_width() + 500, bar.get_y() + bar.get_height()/2,
                f'${value:,.0f}', va='center', fontsize=10)

ax.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.savefig('../outputs/charts/gcc_gdp_per_capita_ranking.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Growth Rate Comparison

In [ ]:
# GDP growth comparison
growth_ind = 'NY.GDP.MKTP.KD.ZG'

growth_data = df[df['indicator_code'] == growth_ind].copy()

fig, ax = plt.subplots(figsize=(14, 7))

for country in growth_data['country_name'].unique():
    country_data = growth_data[growth_data['country_name'] == country].sort_values('year')
    color = colors.get(country, '#999999')
    linewidth = 3 if country == 'Oman' else 1.5
    alpha = 1.0 if country == 'Oman' else 0.7
    
    ax.plot(country_data['year'], country_data['value'],
            marker='o', markersize=3, linewidth=linewidth,
            color=color, alpha=alpha, label=country)

ax.axhline(y=0, color='black', linewidth=0.5)
ax.set_xlabel('Year', fontsize=12)
ax.set_ylabel('GDP Growth Rate (%)', fontsize=12)
ax.set_title('GCC GDP Growth Rate Comparison', fontsize=14, fontweight='bold')
ax.legend(loc='upper right', framealpha=0.9)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../outputs/charts/gcc_growth_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Unemployment Comparison

In [ ]:
# Unemployment rate
unemp_ind = 'SL.UEM.TOTL.ZS'

unemp_data = df[(df['indicator_code'] == unemp_ind) & (df['year'] == latest_year)].copy()
unemp_data = unemp_data.sort_values('value', ascending=False)

fig, ax = plt.subplots(figsize=(10, 6))

bar_colors = [colors.get(c, '#999999') for c in unemp_data['country_name']]
bars = ax.barh(unemp_data['country_name'], unemp_data['value'], color=bar_colors)

ax.set_xlabel('Unemployment Rate (%)', fontsize=12)
ax.set_title(f'GCC Unemployment Rate ({latest_year})', fontsize=14, fontweight='bold')

for bar, value in zip(bars, unemp_data['value']):
    if pd.notna(value):
        ax.text(bar.get_width() + 0.1, bar.get_y() + bar.get_height()/2,
                f'{value:.1f}%', va='center', fontsize=10)

ax.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.savefig('../outputs/charts/gcc_unemployment_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Inflation Comparison

In [ ]:
# Inflation rate
inf_ind = 'FP.CPI.TOTL.ZG'

inf_data = df[df['indicator_code'] == inf_ind].copy()

# Heatmap of inflation over years
pivot = inf_data.pivot_table(index='country_name', columns='year', values='value')

fig, ax = plt.subplots(figsize=(16, 6))

sns.heatmap(pivot, annot=True, fmt='.1f', cmap='RdYlGn_r', center=0,
            ax=ax, cbar_kws={'label': 'Inflation Rate (%)'})

ax.set_title('GCC Inflation Rates Over Time (%)', fontsize=14, fontweight='bold')
ax.set_xlabel('Year', fontsize=12)
ax.set_ylabel('Country', fontsize=12)

plt.tight_layout()
plt.savefig('../outputs/charts/gcc_inflation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Oil Dependency Comparison

In [ ]:
# Oil rents comparison
oil_ind = 'NY.GDP.PETR.RT.ZS'

oil_data = df[(df['indicator_code'] == oil_ind) & (df['year'] == latest_year)].copy()
oil_data = oil_data.sort_values('value', ascending=True)

fig, ax = plt.subplots(figsize=(10, 6))

bar_colors = [colors.get(c, '#999999') for c in oil_data['country_name']]
bars = ax.barh(oil_data['country_name'], oil_data['value'], color=bar_colors)

ax.set_xlabel('Oil Rents (% of GDP)', fontsize=12)
ax.set_title(f'GCC Oil Dependency ({latest_year})', fontsize=14, fontweight='bold')

for bar, value in zip(bars, oil_data['value']):
    if pd.notna(value):
        ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
                f'{value:.1f}%', va='center', fontsize=10)

ax.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.savefig('../outputs/charts/gcc_oil_dependency.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Summary Dashboard

In [ ]:
# Create summary table
indicators = {
    'NY.GDP.MKTP.CD': 'GDP (USD)',
    'NY.GDP.PCAP.CD': 'GDP per Capita (USD)',
    'NY.GDP.MKTP.KD.ZG': 'GDP Growth (%)',
    'SL.UEM.TOTL.ZS': 'Unemployment (%)',
    'FP.CPI.TOTL.ZG': 'Inflation (%)',
    'SP.POP.TOTL': 'Population'
}

summary_data = []

for country in df['country_name'].unique():
    row = {'Country': country}
    for ind_code, ind_name in indicators.items():
        value = df[(df['country_code'] == df[df['country_name'] == country]['country_code'].iloc[0]) &
                   (df['indicator_code'] == ind_code)].sort_values('year', ascending=False)
        if not value.empty and pd.notna(value['value'].iloc[0]):
            row[ind_name] = value['value'].iloc[0]
        else:
            row[ind_name] = None
    summary_data.append(row)

summary_df = pd.DataFrame(summary_data)

# Format the summary
print("\n" + "=" * 80)
print("GCC ECONOMIC SUMMARY")
print("=" * 80)

for _, row in summary_df.iterrows():
    print(f"\n{row['Country']}:")
    if pd.notna(row.get('GDP (USD)')):
        print(f"  GDP: ${row['GDP (USD)']/1e9:.1f} Billion")
    if pd.notna(row.get('GDP per Capita (USD)')):
        print(f"  GDP per Capita: ${row['GDP per Capita (USD)']:,.0f}")
    if pd.notna(row.get('GDP Growth (%)')):
        print(f"  GDP Growth: {row['GDP Growth (%)']:.1f}%")
    if pd.notna(row.get('Population')):
        print(f"  Population: {row['Population']/1e6:.1f} Million")

## 8. Oman's Position in GCC

In [ ]:
print("=" * 60)
print("OMAN'S POSITION IN THE GCC")
print("=" * 60)

# Calculate rankings
rankings = {}

for ind_code, ind_name in indicators.items():
    data = df[(df['indicator_code'] == ind_code) & (df['year'] == df[df['indicator_code'] == ind_code]['year'].max())]
    if not data.empty:
        data = data.sort_values('value', ascending=False).reset_index(drop=True)
        oman_rank = data[data['country_name'] == 'Oman'].index
        if len(oman_rank) > 0:
            rankings[ind_name] = oman_rank[0] + 1

print("\nOman's Rankings (out of 6 GCC countries):")
for metric, rank in rankings.items():
    print(f"  {metric}: #{rank}")

print("\nKey Observations:")
print("  - Oman has the smallest economy in GCC by GDP")
print("  - Middle-ranked in GDP per capita")
print("  - Strong focus on economic diversification")
print("  - Vision 2040 targets sustainable growth")

---

## Analysis Complete

See the `reports/` folder for the comprehensive analysis report.